# Prompt 08 — Google Colab environment verification
**Project:** Token-level uncertainty as a predictor of hallucination in LLMs  
**Status:** Environment smoke test only — **no research results**.

Run the cells **in order** in Google Colab. This notebook checks Python, packages, hardware, the public HaluEval dataset URL, and whether `distilgpt2` can return token logits. **Stop after the last cell and share its complete output.**

We will not compute research metrics or change the frozen experiment protocol in this notebook.

In [ ]:
# CELL 1 — Install the packages needed for the experiment
%pip -q install "transformers>=4.45,<5" "torch>=2.2" "scikit-learn>=1.4" "pandas>=2.0" "numpy>=1.26" "requests>=2.31" "accelerate>=1.0"

In [ ]:
# CELL 2 — Check runtime and package versions
import sys, platform, importlib.metadata as metadata
import torch
print("=== PROMPT 08: ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("OS:", platform.platform())
for name in ["torch", "transformers", "pandas", "numpy", "scikit-learn", "requests", "accelerate"]:
    print(name + ":", metadata.version(name))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("CPU mode: usable for small smoke tests; full scoring may be slow")

In [ ]:
# CELL 3 — Check public dataset accessibility without downloading it all
import requests
url = "https://raw.githubusercontent.com/RUCAIBox/HaluEval/main/data/general_data.json"
try:
    with requests.get(url, stream=True, timeout=25) as response:
        print("Dataset HTTP status:", response.status_code)
        print("Content-Type:", response.headers.get("Content-Type", "unknown"))
        print("Content-Length:", response.headers.get("Content-Length", "unknown"))
        response.raise_for_status()
        first_chunk = next(response.iter_content(chunk_size=2048), b"")
        print("Downloaded initial bytes:", len(first_chunk))
        print("Looks like JSON:", first_chunk.lstrip().startswith((b"[", b"{")))
        print("Dataset access: PASS")
except Exception as exc:
    print("Dataset access: FAIL", type(exc).__name__, str(exc)[:300])
    print("Do not change the dataset yet; share this error.")

In [ ]:
# CELL 4 — Load the proposed scoring model and check token logits
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name = "distilgpt2"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    example = "Question: What is the capital of France?\nAnswer: Paris."
    tokens = tokenizer(example, return_tensors="pt").to(device)
    with torch.inference_mode():
        output = model(**tokens)
    print("Model:", model_name)
    print("Device:", device)
    print("Input token count:", tokens["input_ids"].shape[1])
    print("Logits shape:", tuple(output.logits.shape))
    print("Model smoke test: PASS")
except Exception as exc:
    print("Model smoke test: FAIL", type(exc).__name__, str(exc)[:500])
    print("Share the error before proceeding.")

In [ ]:
# CELL 5 — Print a short checklist to share in ChatGPT
print("\n=== NEXT STEP ===")
print("STOP HERE. Copy the outputs of Cells 2, 3, and 4 into ChatGPT.")
print("If any cell failed, include the complete error message.")
print("No dataset labels, NLL measurements, or final metrics have been computed yet.")